# Titanic Data Analysis

A comprehensive exploratory data analysis of the legendary Titanic dataset — understanding survival patterns, passenger demographics, and the factors that influenced who lived and who perished.

## Project Overview

The RMS Titanic sank on April 15, 1912, after colliding with an iceberg during its maiden voyage. Of the 2,224 passengers and crew aboard, more than 1,500 died, making it one of the deadliest maritime disasters in history.

This notebook analyzes passenger data to uncover patterns in survival rates across different demographic groups, ticket classes, and embarkation points. We explore how factors like gender, age, class, and fare influenced a passenger's chance of survival.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Perform data cleaning on a real-world dataset with missing values
2. Conduct univariate and bivariate analysis on mixed-type features
3. Use cross-tabulations and grouped statistics to find survival patterns
4. Create effective visualizations for categorical and continuous data
5. Apply statistical tests to validate observed differences
6. Extract actionable insights from exploratory analysis

## Business / Research Problem

**Question:** What factors most strongly influenced passenger survival on the Titanic?

Understanding survival patterns helps us explore historical social dynamics (class, gender, age) and how they played out during crisis situations. This dataset is also a classic introduction to data analysis and classification problems.

## Why This Analysis Matters

- **Historical insight:** Reveals societal structures and priorities during the early 20th century
- **Data science education:** One of the most widely used datasets for teaching EDA and ML
- **Pattern discovery:** Demonstrates how to find non-obvious patterns in structured data
- **Safety research:** Lessons applicable to modern disaster preparedness and evacuation protocols

## Dataset Overview

The Titanic dataset contains information about passengers aboard the Titanic.

**Key columns:**

| Column | Description |
|--------|-------------|
| PassengerId | Unique identifier |
| Survived | 0 = No, 1 = Yes |
| Pclass | Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd |
| Name | Passenger name |
| Sex | Gender |
| Age | Age in years |
| SibSp | # of siblings/spouses aboard |
| Parch | # of parents/children aboard |
| Ticket | Ticket number |
| Fare | Passenger fare |
| Cabin | Cabin number |
| Embarked | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

## Dataset Source and License Notes

- **Source:** [Kaggle Titanic Competition](https://www.kaggle.com/competitions/titanic)
- **License:** Public domain / competition dataset
- **Citation:** This dataset is based on the Encyclopedia Titanica passenger records

## Environment Setup

Install required packages if not already available.

In [1]:
!pip install -q pandas numpy matplotlib seaborn scipy kaggle

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


## Configuration / Constants

In [3]:
# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Random seed
RANDOM_STATE = 42

# Data path
DATA_DIR = 'data'

## Dataset Download

We download the Titanic dataset directly from Kaggle using the Kaggle API. Make sure your `kaggle.json` credentials file is properly set up.

In [4]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle competitions download -c titanic -p {DATA_DIR} --force

# Extract if zipped
import zipfile
zip_path = os.path.join(DATA_DIR, 'titanic.zip')
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)

print("Files in data directory:")
for f in os.listdir(DATA_DIR):
    print(f"  {f}")


Files in data directory:
  gender_submission.csv
  test.csv
  train.csv


C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(

  0%|          | 0.00/34.1k [00:00<?, ?B/s]
100%|##########| 34.1k/34.1k [00:00<00:00, 159kB/s]
100%|##########| 34.1k/34.1k [00:00<00:00, 159kB/s]


## Data Loading

We load the training dataset which contains the `Survived` column (the test set does not).

In [5]:
df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print(f"Dataset shape: {df.shape}")
print(f"Number of passengers: {len(df)}")
df.head()

Dataset shape: (891, 12)
Number of passengers: 891


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Data Validation Checks

Before analysis, let's validate the dataset for completeness, types, and anomalies.

In [6]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Count'] > 0])

print("\n=== Basic Statistics ===")
df.describe()

=== Data Types ===
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

=== Missing Values ===
          Count  Percentage
Age         177       19.87
Cabin       687       77.10
Embarked      2        0.22

=== Basic Statistics ===


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [7]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate PassengerIds: {df['PassengerId'].duplicated().sum()}")

# Validate Survived column
print(f"\nSurvived values: {df['Survived'].unique()}")
print(f"Pclass values: {sorted(df['Pclass'].unique())}")
print(f"Sex values: {df['Sex'].unique()}")
print(f"Embarked values: {df['Embarked'].unique()}")

Duplicate rows: 0
Duplicate PassengerIds: 0

Survived values: [0 1]
Pclass values: [np.int64(1), np.int64(2), np.int64(3)]
Sex values: ['male' 'female']
Embarked values: ['S' 'C' 'Q' nan]


## Data Cleaning

We handle missing values and create useful derived features.

In [8]:
# Fill Age with median grouped by Pclass and Sex (more accurate than overall median)
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

# Fill Embarked with mode (only 2 missing)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin has too many missing values — extract deck letter instead
df['Deck'] = df['Cabin'].str[0]
df['HasCabin'] = df['Cabin'].notna().astype(int)

# Create family size feature
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Extract title from name
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Age group
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 55, 80],
                        labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])

print("Missing values after cleaning:")
print(df[['Age', 'Embarked', 'Fare']].isnull().sum())
print(f"\nNew columns created: Deck, HasCabin, FamilySize, IsAlone, Title, AgeGroup")

Missing values after cleaning:
Age         0
Embarked    0
Fare        0
dtype: int64

New columns created: Deck, HasCabin, FamilySize, IsAlone, Title, AgeGroup


## Exploratory Data Analysis

Let's start with an overall view of survival and passenger composition.

In [9]:
print("=== Survival Distribution ===")
survived_counts = df['Survived'].value_counts()
print(f"Died: {survived_counts[0]} ({survived_counts[0]/len(df)*100:.1f}%)")
print(f"Survived: {survived_counts[1]} ({survived_counts[1]/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Survival count
sns.countplot(data=df, x='Survived', ax=axes[0])
axes[0].set_xticklabels(['Died', 'Survived'])
axes[0].set_title('Survival Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom')

# Survival percentage
axes[1].pie(survived_counts, labels=['Died', 'Survived'], autopct='%1.1f%%',
            colors=['#ff6b6b', '#51cf66'], startangle=90)
axes[1].set_title('Survival Rate')

plt.tight_layout()
plt.show()

=== Survival Distribution ===
Died: 549 (61.6%)
Survived: 342 (38.4%)


## Univariate Analysis

Examining each key variable independently before looking at interactions.

In [10]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Age distribution
axes[0, 0].hist(df['Age'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')
axes[0, 0].axvline(df['Age'].median(), color='red', linestyle='--', label=f'Median: {df["Age"].median():.0f}')
axes[0, 0].legend()

# Fare distribution
axes[0, 1].hist(df['Fare'], bins=40, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Fare Distribution')
axes[0, 1].set_xlabel('Fare (£)')

# Pclass distribution
sns.countplot(data=df, x='Pclass', ax=axes[0, 2])
axes[0, 2].set_title('Passenger Class')

# Sex distribution
sns.countplot(data=df, x='Sex', ax=axes[1, 0])
axes[1, 0].set_title('Gender Distribution')

# Embarked distribution
sns.countplot(data=df, x='Embarked', ax=axes[1, 1])
axes[1, 1].set_title('Embarkation Port')

# Family size
sns.countplot(data=df, x='FamilySize', ax=axes[1, 2])
axes[1, 2].set_title('Family Size')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

Now let's explore how survival relates to each key feature.

In [11]:
# Survival by gender
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender vs Survival
survival_by_sex = df.groupby('Sex')['Survived'].mean()
survival_by_sex.plot(kind='bar', ax=axes[0], color=['#4ecdc4', '#ff6b6b'])
axes[0].set_title('Survival Rate by Gender')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
for i, v in enumerate(survival_by_sex):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

# Class vs Survival
survival_by_class = df.groupby('Pclass')['Survived'].mean()
survival_by_class.plot(kind='bar', ax=axes[1], color=['#ffd93d', '#6bcb77', '#ff6b6b'])
axes[1].set_title('Survival Rate by Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)
for i, v in enumerate(survival_by_class):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)

# Embarked vs Survival
survival_by_embarked = df.groupby('Embarked')['Survived'].mean()
survival_by_embarked.plot(kind='bar', ax=axes[2], color=['#a8dadc', '#457b9d', '#1d3557'])
axes[2].set_title('Survival Rate by Embarkation Port')
axes[2].set_ylabel('Survival Rate')
axes[2].set_ylim(0, 1)
for i, v in enumerate(survival_by_embarked):
    axes[2].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [12]:
# Survival by class AND gender — the most revealing cross-tabulation
ct = pd.crosstab(df['Pclass'], df['Sex'], values=df['Survived'], aggfunc='mean').round(3)
print("Survival Rate by Class and Gender:")
print(ct)

fig, ax = plt.subplots(figsize=(8, 5))
ct.plot(kind='bar', ax=ax)
ax.set_title('Survival Rate by Class and Gender')
ax.set_ylabel('Survival Rate')
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Gender')
plt.tight_layout()
plt.show()

Survival Rate by Class and Gender:
Sex     female   male
Pclass               
1        0.968  0.369
2        0.921  0.157
3        0.500  0.135


In [13]:
# Age vs Survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution by survival
for survived, label in [(0, 'Died'), (1, 'Survived')]:
    axes[0].hist(df[df['Survived'] == survived]['Age'].dropna(), bins=30, alpha=0.6, label=label)
axes[0].set_title('Age Distribution by Survival')
axes[0].set_xlabel('Age')
axes[0].legend()

# Survival by age group
survival_by_age = df.groupby('AgeGroup', observed=False)['Survived'].mean()
survival_by_age.plot(kind='bar', ax=axes[1], color='#4ecdc4')
axes[1].set_title('Survival Rate by Age Group')
axes[1].set_ylabel('Survival Rate')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(survival_by_age):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

Diving deeper into individual features and their relationship with survival.

In [14]:
# Family size vs survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

survival_by_family = df.groupby('FamilySize')['Survived'].mean()
survival_by_family.plot(kind='bar', ax=axes[0], color='#6bcb77')
axes[0].set_title('Survival Rate by Family Size')
axes[0].set_ylabel('Survival Rate')
axes[0].tick_params(axis='x', rotation=0)

# Alone vs Not Alone
survival_by_alone = df.groupby('IsAlone')['Survived'].mean()
survival_by_alone.index = ['With Family', 'Alone']
survival_by_alone.plot(kind='bar', ax=axes[1], color=['#4ecdc4', '#ff6b6b'])
axes[1].set_title('Survival Rate: Alone vs With Family')
axes[1].set_ylabel('Survival Rate')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(survival_by_alone):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [15]:
# Title analysis
title_survival = df.groupby('Title').agg(
    Count=('Survived', 'count'),
    SurvivalRate=('Survived', 'mean')
).sort_values('Count', ascending=False)

print("Survival by Title (top 10):")
print(title_survival.head(10).to_string())

# Fare vs survival
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='Survived', y='Fare', ax=ax)
ax.set_xticklabels(['Died', 'Survived'])
ax.set_title('Fare Distribution by Survival')
ax.set_ylim(0, 300)  # Cap for visibility
plt.tight_layout()
plt.show()

Survival by Title (top 10):
        Count  SurvivalRate
Title                      
Mr        517      0.156673
Miss      182      0.697802
Mrs       125      0.792000
Master     40      0.575000
Dr          7      0.428571
Rev         6      0.000000
Mlle        2      1.000000
Major       2      0.500000
Col         2      0.500000
Capt        1      0.000000


## Statistical Checks / Hypothesis Testing

Let's formally test whether the observed survival differences are statistically significant.

In [16]:
# Chi-squared test: Gender vs Survival
contingency_sex = pd.crosstab(df['Sex'], df['Survived'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_sex)
print(f"Chi-squared test: Gender vs Survival")
print(f"  Chi² = {chi2:.4f}, p-value = {p_val:.2e}")
print(f"  {'Significant' if p_val < 0.05 else 'Not significant'} at α=0.05")

print()

# Chi-squared test: Class vs Survival
contingency_class = pd.crosstab(df['Pclass'], df['Survived'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_class)
print(f"Chi-squared test: Class vs Survival")
print(f"  Chi² = {chi2:.4f}, p-value = {p_val:.2e}")
print(f"  {'Significant' if p_val < 0.05 else 'Not significant'} at α=0.05")

print()

# T-test: Age of survived vs died
survived_ages = df[df['Survived'] == 1]['Age'].dropna()
died_ages = df[df['Survived'] == 0]['Age'].dropna()
t_stat, p_val = stats.ttest_ind(survived_ages, died_ages)
print(f"T-test: Age (Survived vs Died)")
print(f"  t = {t_stat:.4f}, p-value = {p_val:.4f}")
print(f"  Mean age survived: {survived_ages.mean():.1f}, died: {died_ages.mean():.1f}")
print(f"  {'Significant' if p_val < 0.05 else 'Not significant'} at α=0.05")

print()

# Mann-Whitney U test: Fare of survived vs died
survived_fares = df[df['Survived'] == 1]['Fare']
died_fares = df[df['Survived'] == 0]['Fare']
u_stat, p_val = stats.mannwhitneyu(survived_fares, died_fares, alternative='two-sided')
print(f"Mann-Whitney U test: Fare (Survived vs Died)")
print(f"  U = {u_stat:.4f}, p-value = {p_val:.2e}")
print(f"  Median fare survived: £{survived_fares.median():.2f}, died: £{died_fares.median():.2f}")
print(f"  {'Significant' if p_val < 0.05 else 'Not significant'} at α=0.05")

Chi-squared test: Gender vs Survival
  Chi² = 260.7170, p-value = 1.20e-58
  Significant at α=0.05

Chi-squared test: Class vs Survival
  Chi² = 102.8890, p-value = 4.55e-23
  Significant at α=0.05

T-test: Age (Survived vs Died)
  t = -1.7796, p-value = 0.0755
  Mean age survived: 28.1, died: 29.7
  Not significant at α=0.05

Mann-Whitney U test: Fare (Survived vs Died)
  U = 129951.5000, p-value = 4.55e-22
  Median fare survived: £26.00, died: £10.50
  Significant at α=0.05


## Visual Analysis

A comprehensive visual summary of the key relationships in the data.

In [17]:
# Correlation heatmap
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'HasCabin']
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [18]:
# Comprehensive survival heatmap by class, gender, and embarkation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival by Class and Gender
pivot1 = df.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean')
sns.heatmap(pivot1, annot=True, fmt='.2%', cmap='RdYlGn', ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Survival Rate: Class × Gender')

# Survival by Class and Embarked
pivot2 = df.pivot_table(values='Survived', index='Pclass', columns='Embarked', aggfunc='mean')
sns.heatmap(pivot2, annot=True, fmt='.2%', cmap='RdYlGn', ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Survival Rate: Class × Embarkation')

plt.tight_layout()
plt.show()

In [19]:
# Violin plot: Age by class and survival
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df, x='Pclass', y='Age', hue='Survived', split=True, ax=ax, palette=['#ff6b6b', '#51cf66'])
ax.set_title('Age Distribution by Class and Survival')
ax.legend(title='Survived', labels=['Died', 'Survived'])
plt.tight_layout()
plt.show()

## Key Findings

Based on the analysis above:

1. **Gender was the strongest survival predictor** — Female passengers had dramatically higher survival rates than males, consistent with the "women and children first" protocol.

2. **Passenger class strongly influenced survival** — 1st class passengers survived at much higher rates than 3rd class, reflecting both cabin location (closer to lifeboats) and social privilege.

3. **Children had better survival odds** — The youngest passengers had elevated survival rates, especially in 1st and 2nd class.

4. **Fare correlates with survival** — Higher fares (linked to higher class) were associated with better survival outcomes.

5. **Family size matters** — Passengers with small families (2-4 members) survived at higher rates than solo travelers or very large families.

6. **Embarkation port shows differences** — Cherbourg passengers had higher survival rates, likely due to a higher proportion of 1st class passengers boarding there.

7. **Having a cabin recorded** correlates with survival — likely a proxy for class and fare level.

## Limitations

- **Incomplete data:** ~20% of Age values and ~77% of Cabin values were missing, requiring imputation which introduces uncertainty
- **Training set only:** We analyzed 891 of the ~2,224 total passengers — the test set lacks survival labels
- **No crew data:** The dataset only covers passengers, not crew members
- **Correlation ≠ causation:** Class correlated with survival, but the causal mechanism (proximity to lifeboats, social status, etc.) isn't captured in the data
- **Imputed ages:** Group-based median imputation may mask true age-survival relationships

## Recommendations / Next Steps

1. **Build a predictive model** — Use logistic regression, random forest, or gradient boosting to predict survival
2. **Feature engineering** — Extract more info from Cabin (deck), Name (title), and Ticket (shared tickets)
3. **Cross-validate** — Use proper k-fold CV to evaluate model performance
4. **Include test set** — Submit predictions to Kaggle to benchmark against other approaches
5. **Compare imputation strategies** — Test KNN imputation, MICE, or model-based approaches for Age

## Common Mistakes

1. **Dropping Age rows entirely** — This loses ~20% of data. Better to impute thoughtfully.
2. **Ignoring Cabin** — While mostly missing, the cabin deck letter and presence/absence are informative.
3. **Using test set for EDA** — The test set has no survival labels, so it can't be used for analysis.
4. **Not grouping rare titles** — Titles like 'Mlle', 'Mme', 'Countess' should be consolidated.
5. **Treating Pclass as continuous** — While numbered 1-3, it's ordinal, not ratio-scaled.
6. **Ignoring feature interactions** — Gender × Class interaction is much more informative than either alone.
7. **Data leakage** — Fitting imputers on the entire dataset before train-test split.

## Mini Challenge / Exercises

Test your understanding with these exercises:

1. **Deck analysis:** For passengers with known cabin numbers, calculate survival rate by deck (A-G). Which deck had the best survival rate?

2. **Shared tickets:** Identify passengers who shared the same ticket number. Did traveling on a shared ticket improve survival odds?

3. **Name length:** Is there any correlation between name length and survival? (Hint: this might be a proxy for social status/title length)

4. **Fare outliers:** Identify passengers who paid extremely high fares (>3 standard deviations above mean). What's their survival rate?

5. **Build a baseline model:** Using only Sex and Pclass, build a simple rule-based predictor. What accuracy do you achieve?

In [20]:
# Space for your exercises
# Exercise 1: Deck analysis
# deck_survival = df.groupby('Deck')['Survived'].agg(['mean', 'count'])
# print(deck_survival)

# Exercise 2: Shared tickets
# ticket_counts = df['Ticket'].value_counts()
# shared_tickets = ticket_counts[ticket_counts > 1].index
# ...

print("Uncomment the exercises above and try them!")

Uncomment the exercises above and try them!


## Final Summary / Key Takeaways

| Factor | Impact on Survival |
|--------|--------------------|
| Gender | Female >> Male |
| Class | 1st > 2nd > 3rd |
| Age | Children favored |
| Fare | Higher fare = better odds |
| Family | Small families best |
| Embarked | Cherbourg highest rate |

**The Titanic disaster starkly revealed social hierarchies.** Gender and class were the dominant factors in survival, reflecting the "women and children first" evacuation protocol and the preferential access to lifeboats that upper-class passengers had.

This dataset remains an excellent introduction to data analysis because it contains mixed data types, missing values, feature engineering opportunities, and genuinely interesting patterns that connect to real historical events.